In [2]:
import requests
from bs4 import BeautifulSoup
import time
import csv
import re
from urllib.parse import urljoin, urlparse

def scrape_despegartickets(url_base, limite_registros=300):
    """
    Scraper para Despegar.com con paginación
    """
    resultados = []
    pagina_actual = 1
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KDE, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'es-ES,es;q=0.8,en-US;q=0.5,en;q=0.3',
        'Accept-Encoding': 'gzip, deflate',
        'Connection': 'keep-alive',
    }
    
    while len(resultados) < limite_registros:
        # Construir URL con paginación (ajustar según la estructura real)
        if pagina_actual == 1:
            url = url_base
        else:
            # Formato común: /page/X/ o ?page=X
            url = url_base + f"?page={pagina_actual}"
        
        print(f"Scraping página {pagina_actual}: {url}")
        
        try:
            response = requests.get(url, headers=headers, timeout=15)
            
            if response.status_code != 200:
                print(f"Error {response.status_code} en página {pagina_actual}")
                break
                
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # SELECTORES - ¡DEBES ACTUALIZAR ESTOS!
            # Inspecciona Despegar para obtener las clases reales:
            # - Busca el contenedor de cada resultado (hotel/vuelo)
            # - Busca la etiqueta del nombre
            # - Busca la etiqueta del precio
            
            # EJEMPLO (necesitas adaptarlo):
            items = soup.find_all('div', class_='item-resultado')  # <-- CAMBIAR
            
            if not items:
                print("No se encontraron más resultados")
                break
                
            for item in items:
                if len(resultados) >= limite_registros:
                    break
                    
                # Extraer nombre (ejemplo)
                nombre_tag = item.find('h3', class_='nombre-producto')  # <-- CAMBIAR
                nombre = nombre_tag.text.strip() if nombre_tag else "N/A"
                
                # Extraer precio (ejemplo)
                precio_tag = item.find('span', class_='precio')  # <-- CAMBIAR
                if precio_tag:
                    precio_texto = precio_tag.text.strip()
                    # Limpiar precio (remover símbolos de moneda)
                    precio_limpio = re.sub(r'[^\d.,]', '', precio_texto)
                else:
                    precio_limpio = "N/A"
                
                resultados.append({
                    'nombre': nombre,
                    'precio': precio_limpio,
                    'pagina': pagina_actual
                })
                
                print(f"  [{len(resultados)}] {nombre[:50]} - {precio_limpio}")
            
            # Verificar si hay siguiente página
            # Buscar botón/liga "Siguiente"
            next_button = soup.find('a', string=re.compile(r'Siguiente|Next|→'))
            if not next_button:
                print("No hay más páginas disponibles")
                break
                
            pagina_actual += 1
            time.sleep(3)  # Delay para no saturar el servidor
            
        except requests.exceptions.RequestException as e:
            print(f"Error de conexión: {e}")
            break
        except Exception as e:
            print(f"Error inesperado: {e}")
            break
    
    print(f"\n✅ Scraping completado: {len(resultados)} registros obtenidos")
    return resultados

def guardar_csv(datos, archivo="despegar_precios.csv"):
    """Guarda los resultados en CSV"""
    if not datos:
        print("No hay datos para guardar")
        return
    
    with open(archivo, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['nombre', 'precio', 'pagina'])
        writer.writeheader()
        writer.writerows(datos)
    
    print(f"📁 Datos guardados en {archivo}")

# ========== CONFIGURACIÓN ==========
if __name__ == "__main__":
    # URL DE BÚSQUEDA DE DESPEGAR
    # Ejemplo: búsqueda de hoteles en Cancún
    URL_BASE = "https://www.despegar.com.ar/shop/hoteles/search/..."  # <--- ¡CAMBIA ESTO!
    
    print("⚠️  IMPORTANTE: Antes de ejecutar, actualiza los selectores CSS")
    print("   y la URL de búsqueda en el código.\n")
    
    # Ejecutar scraping
    resultados = scrape_despegartickets(URL_BASE, limite_registros=300)
    
    # Guardar resultados
    guardar_csv(resultados)

⚠️  IMPORTANTE: Antes de ejecutar, actualiza los selectores CSS
   y la URL de búsqueda en el código.

Scraping página 1: https://www.despegar.com.ar/shop/hoteles/search/...
Error 404 en página 1

✅ Scraping completado: 0 registros obtenidos
No hay datos para guardar
